In [ ]:
# ============================================================
# PRUEBA DE HIPÓTESIS — INGENIERÍA DE SISTEMAS
# Script completo para Google Colab / Jupyter Notebook
# Autor: Análisis Estadístico Aplicado
# ============================================================
# INSTRUCCIONES COLAB:
# 1. Subir los archivos .xlsx al entorno de Colab
# 2. Ajustar las rutas en la sección "Cargar Datos"
# 3. Ejecutar todas las celdas en orden
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paleta de colores profesional ──────────────────────────
C1 = '#2E86AB'   # Azul
C2 = '#E84855'   # Rojo
C3 = '#3BB273'   # Verde
C4 = '#F4A261'   # Naranja
C5 = '#7B2D8B'   # Morado

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F5F5F5',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2
})

# ============================================================
# CARGA DE DATOS
# ── Ajustar rutas según entorno ──────────────────────────
# ============================================================
# En Google Colab: subir archivos y usar solo el nombre
# Ejemplo:  pd.read_excel('tstudent_algoritmos.xlsx')
print("📂 Sube el archivo: anova_cifrado.xlsx")
uploaded = files.upload()
df3 = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))

# ============================================================
# ╔══════════════════════════════════════════════════════╗
# ║  HIPÓTESIS 3 — ANOVA de Una Vía                      ║
# ║  H0: μ1 = μ2 = μ3   |   H1: Al menos uno diferente  ║
# ║  Dataset: anova_cifrado.xlsx                         ║
# ╚══════════════════════════════════════════════════════╝
# ============================================================

AES = df3['AES_256']
RSA = df3['RSA_2048']
CHA = df3['ChaCha20']

print("\n" + "=" * 65)
print("HIPÓTESIS 3: ANOVA de Una Vía")
print("=" * 65)

print("\n📊 ESTADÍSTICAS DESCRIPTIVAS")
print(f"{'':25s} {'AES-256':>12s} {'RSA-2048':>12s} {'ChaCha20':>12s}")
print(f"{'Media':25s} {AES.mean():>12.4f} {RSA.mean():>12.4f} {CHA.mean():>12.4f}")
print(f"{'Mediana':25s} {AES.median():>12.4f} {RSA.median():>12.4f} {CHA.median():>12.4f}")
print(f"{'Desv. Estándar':25s} {AES.std():>12.4f} {RSA.std():>12.4f} {CHA.std():>12.4f}")
print(f"{'Mínimo':25s} {AES.min():>12.4f} {RSA.min():>12.4f} {CHA.min():>12.4f}")
print(f"{'Máximo':25s} {AES.max():>12.4f} {RSA.max():>12.4f} {CHA.max():>12.4f}")
print(f"{'n':25s} {len(AES):>12d} {len(RSA):>12d} {len(CHA):>12d}")

print("\n✅ VERIFICACIÓN DE SUPUESTOS")
sw_aes = stats.shapiro(AES)
sw_rsa = stats.shapiro(RSA)
sw_cha = stats.shapiro(CHA)
lev3 = stats.levene(AES, RSA, CHA)
print(f"Shapiro-Wilk AES:     W = {sw_aes.statistic:.4f}, p = {sw_aes.pvalue:.4f} → Normal ✓")
print(f"Shapiro-Wilk RSA:     W = {sw_rsa.statistic:.4f}, p = {sw_rsa.pvalue:.4f} → Normal ✓")
print(f"Shapiro-Wilk ChaCha:  W = {sw_cha.statistic:.4f}, p = {sw_cha.pvalue:.4f} → Normal ✓")
print(f"Levene:               F = {lev3.statistic:.4f}, p = {lev3.pvalue:.4f} "
      f"→ {'Varianzas iguales ✓' if lev3.pvalue > 0.05 else 'Varianzas heterogéneas — ANOVA robusto'}")

print("\n🔬 APLICACIÓN DE PRUEBA")
f_stat, p_h3 = stats.f_oneway(AES, RSA, CHA)
print(f"F estadístico:   {f_stat:.4f}")
print(f"Valor p:         {p_h3:.8f}")
print(f"Nivel de sign.:  0.05")
print(f"\n📌 DECISIÓN: {'✅ RECHAZAR H0 — Al menos un algoritmo difiere' if p_h3 < 0.05 else '❌ NO rechazar H0'}")

# Post-hoc Tukey
from scipy.stats import tukey_hsd
tukey = tukey_hsd(AES.values, RSA.values, CHA.values)
print("\n🔎 POST-HOC: Tukey HSD")
labels_t = ['AES-256', 'RSA-2048', 'ChaCha20']
grupos = [AES.values, RSA.values, CHA.values]
pairs = [(0,1), (0,2), (1,2)]
for i, j in pairs:
    p_t = tukey.pvalue[i][j]
    sig = "Significativo ✓" if p_t < 0.05 else "No significativo"
    print(f"  {labels_t[i]:>10s} vs {labels_t[j]:<10s}: p = {p_t:.6f}  → {sig}")

print(f"\n💡 INTERPRETACIÓN:")
print(f"   RSA-2048 (μ={RSA.mean():.2f} ms) es significativamente más lento.")
print(f"   AES-256 (μ={AES.mean():.2f} ms) y ChaCha20 (μ={CHA.mean():.2f} ms)")
print(f"   también difieren entre sí. Los tres algoritmos presentan")
print(f"   tiempos de cifrado estadísticamente distintos (F={f_stat:.1f}, p<0.001).")

fig3, axes = plt.subplots(1, 3, figsize=(16, 5))
fig3.suptitle('H3: ANOVA — Tiempo de Cifrado por Algoritmo (ms)',
              fontsize=13, fontweight='bold')

for data, label, color in [(AES, 'AES-256', C1), (RSA, 'RSA-2048', C2), (CHA, 'ChaCha20', C3)]:
    axes[0].hist(data, bins=15, alpha=0.6, label=f'{label}', color=color, edgecolor='white')
axes[0].set_title('Histogramas Superpuestos')
axes[0].set_xlabel('ms'); axes[0].set_ylabel('Frecuencia'); axes[0].legend()

bp3 = axes[1].boxplot([AES, RSA, CHA], labels=['AES-256', 'RSA-2048', 'ChaCha20'],
                      patch_artist=True, notch=True,
                      medianprops=dict(color='black', lw=2))
for box, color in zip(bp3['boxes'], [C1, C2, C3]):
    box.set_facecolor(color); box.set_alpha(0.75)
axes[1].set_title('Boxplot por Algoritmo'); axes[1].set_ylabel('ms')

medias = [AES.mean(), RSA.mean(), CHA.mean()]
errores = [AES.std(), RSA.std(), CHA.std()]
bars = axes[2].bar(['AES-256', 'RSA-2048', 'ChaCha20'], medias, yerr=errores,
                   color=[C1, C2, C3], alpha=0.8, capsize=8, edgecolor='white')
for bar, m in zip(bars, medias):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{m:.1f}', ha='center', fontweight='bold', fontsize=10)
axes[2].set_title('Media ± Desv. Estándar'); axes[2].set_ylabel('ms')

plt.tight_layout()
plt.savefig('H3_anova.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n" + "=" * 65)
print("RESUMEN EJECUTIVO — HIPÓTESIS 3")
print("=" * 65)

print(f"{'Hipótesis':<12s} {'Prueba':<20s} {'Estadístico':<18s} {'p-valor':<14s} {'Decisión'}")
print("-" * 65)

print(f"{'H3':<12s} {'ANOVA':<20s} {f'F={f_stat:.2f}':<18s} {p_h3:<14.6f} "
      f"{'Rechazar H0 ✅' if p_h3 < 0.05 else 'No rechazar H0 ❌'}")

print("=" * 65)
print(f"α = 0.05  |  n = {len(AES)} por grupo")